# Statistics for Bioinformatics

**Tier 3 -- Applied Bioinformatics**

Bioinformatics analyses generate thousands to millions of measurements. Proper statistical methods are essential to distinguish genuine biological signals from noise. This notebook covers the statistical toolkit every bioinformatician needs: from hypothesis testing and multiple testing correction to survival analysis.

**Prerequisites:** Tier 2 (basic Python, NumPy, pandas)  
**Libraries:** `numpy`, `pandas`, `scipy.stats`, `statsmodels`, `matplotlib`

---[< Previous: Promoter and Regulatory Sequence Analysis](../05_Promoter_and_Regulatory_Analysis/02_regulatory_analysis.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Machine Learning for Biology >](../07_Machine_Learning_for_Biology/01_machine_learning_for_biology.ipynb)---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 1. Distributions in Biology

Choosing the right statistical test starts with understanding your data's distribution.

| Distribution | Use in biology | Example |
|-------------|---------------|--------|
| **Normal** | Continuous measurements | Height, weight, log-transformed expression |
| **Poisson** | Count data (rare events) | Mutations per gene, read counts |
| **Negative binomial** | Overdispersed counts | RNA-seq read counts (DESeq2 uses this) |
| **Binomial** | Success/failure in n trials | SNPs in a region, methylated CpGs |
| **Exponential** | Waiting times | Distance between mutations |
| **Uniform** | p-values under the null | Expected distribution of p-values when H0 is true |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Normal -- gene expression (log2)
x_norm = np.random.normal(8, 2, 5000)
axes[0, 0].hist(x_norm, bins=50, color='steelblue', edgecolor='white', density=True)
axes[0, 0].set_title('Normal\n(Log2 expression)')

# Poisson -- mutations per gene
x_pois = np.random.poisson(3, 5000)
axes[0, 1].hist(x_pois, bins=range(15), color='coral', edgecolor='white', density=True, align='left')
axes[0, 1].set_title('Poisson\n(Mutations per gene)')

# Negative binomial -- RNA-seq counts
x_nb = np.random.negative_binomial(5, 0.01, 5000)
axes[0, 2].hist(x_nb, bins=50, color='mediumseagreen', edgecolor='white', density=True)
axes[0, 2].set_title('Negative Binomial\n(RNA-seq read counts)')

# Binomial -- methylated CpGs out of 20
x_binom = np.random.binomial(20, 0.3, 5000)
axes[1, 0].hist(x_binom, bins=range(22), color='mediumpurple', edgecolor='white', density=True, align='left')
axes[1, 0].set_title('Binomial\n(Methylated CpGs / 20 sites)')

# Exponential -- inter-mutation distance
x_exp = np.random.exponential(1000, 5000)
axes[1, 1].hist(x_exp, bins=50, color='goldenrod', edgecolor='white', density=True)
axes[1, 1].set_title('Exponential\n(Distance between mutations, bp)')

# Uniform -- p-values under null
x_unif = np.random.uniform(0, 1, 5000)
axes[1, 2].hist(x_unif, bins=50, color='gray', edgecolor='white', density=True)
axes[1, 2].set_title('Uniform\n(p-values under H0)')

for ax in axes.flat:
    ax.set_ylabel('Density')

plt.suptitle('Common Distributions in Bioinformatics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Hypothesis Testing Review

### The framework

1. **Null hypothesis (H0)**: No effect / no difference (e.g., gene expression is the same in treated and control)
2. **Alternative hypothesis (H1)**: There is an effect
3. **Test statistic**: Summarizes the data (e.g., t-statistic, chi-squared)
4. **p-value**: Probability of observing the test statistic (or more extreme) if H0 is true
5. **Decision**: Reject H0 if p-value < significance level (typically 0.05)

### Type I and Type II errors

| | H0 is true | H0 is false |
|---|---|---|
| **Reject H0** | Type I error (false positive, rate = alpha) | Correct (power = 1 - beta) |
| **Fail to reject H0** | Correct | Type II error (false negative, rate = beta) |

In [ ]:
# Demonstration: t-test on gene expression data
np.random.seed(42)

# Simulated gene expression (log2 scale): treated vs control
control = np.random.normal(loc=8.0, scale=1.5, size=30)
treated = np.random.normal(loc=9.2, scale=1.5, size=30)  # 1.2-fold increase in log2

t_stat, p_val = stats.ttest_ind(treated, control)

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([control, treated], labels=['Control', 'Treated'], widths=0.5)
ax.scatter(np.ones(30) + np.random.normal(0, 0.03, 30), control, alpha=0.5, color='steelblue')
ax.scatter(2 * np.ones(30) + np.random.normal(0, 0.03, 30), treated, alpha=0.5, color='coral')
ax.set_ylabel('Log2 expression')
ax.set_title(f'Gene Expression: Control vs Treated\nt={t_stat:.2f}, p={p_val:.4f}')
plt.tight_layout()
plt.show()

print(f"Control: mean={control.mean():.2f}, std={control.std():.2f}")
print(f"Treated: mean={treated.mean():.2f}, std={treated.std():.2f}")
print(f"Welch's t-test: t={t_stat:.3f}, p={p_val:.4e}")

---
## 3. The Multiple Testing Problem

### Why it matters in genomics

In a typical RNA-seq experiment, you test ~20,000 genes for differential expression. If you use alpha = 0.05:

$$\text{Expected false positives} = 20{,}000 \times 0.05 = 1{,}000 \text{ genes!}$$

You would report 1,000 genes as differentially expressed purely by chance. This is why **multiple testing correction** is critical in genomics.

In [ ]:
# Demonstrate the multiple testing problem
np.random.seed(42)

n_genes = 20000
n_true_de = 500       # truly differentially expressed genes
n_null = n_genes - n_true_de

# Generate p-values
# Null genes: p-values are uniform
p_null = np.random.uniform(0, 1, n_null)
# True DE genes: p-values are small (drawn from beta distribution)
p_de = np.random.beta(0.3, 5, n_true_de)

p_values = np.concatenate([p_null, p_de])
is_true_de = np.array([False] * n_null + [True] * n_true_de)

# Without correction: how many false positives?
alpha = 0.05
sig_uncorrected = p_values < alpha
fp_uncorrected = sig_uncorrected & ~is_true_de
tp_uncorrected = sig_uncorrected & is_true_de

print(f"Total genes tested: {n_genes}")
print(f"Truly DE genes: {n_true_de}")
print(f"\nWithout correction (alpha = {alpha}):")
print(f"  Significant calls: {sig_uncorrected.sum()}")
print(f"  True positives: {tp_uncorrected.sum()}")
print(f"  FALSE POSITIVES: {fp_uncorrected.sum()}")
print(f"  False discovery rate: {fp_uncorrected.sum() / max(sig_uncorrected.sum(), 1):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# P-value distribution
axes[0].hist(p_null, bins=50, alpha=0.7, color='gray', label=f'Null genes (n={n_null})', density=True)
axes[0].hist(p_de, bins=50, alpha=0.7, color='red', label=f'True DE genes (n={n_true_de})', density=True)
axes[0].axvline(0.05, color='black', linestyle='--', label='alpha = 0.05')
axes[0].set_xlabel('p-value')
axes[0].set_ylabel('Density')
axes[0].set_title('P-value Distributions')
axes[0].legend()

# Combined histogram (what you actually see)
axes[1].hist(p_values, bins=50, color='steelblue', edgecolor='white', density=True)
axes[1].axvline(0.05, color='red', linestyle='--', linewidth=2, label='alpha = 0.05')
axes[1].set_xlabel('p-value')
axes[1].set_ylabel('Density')
axes[1].set_title('Combined P-value Distribution (all 20,000 genes)\nNote the spike near 0 = true signal')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. Multiple Testing Corrections

### 4.1 Bonferroni correction

The simplest and most conservative approach: divide alpha by the number of tests.

$$\alpha_{\text{Bonferroni}} = \frac{\alpha}{m}$$

For 20,000 genes at alpha = 0.05: $\alpha_{\text{Bonf}} = 2.5 \times 10^{-6}$

**Pros:** Controls the family-wise error rate (FWER) -- the probability of even one false positive.  
**Cons:** Very conservative. Many true positives are missed.

### 4.2 Benjamini-Hochberg (FDR control)

Controls the **false discovery rate (FDR)** -- the expected proportion of false positives among all discoveries.

Algorithm:
1. Sort p-values: $p_{(1)} \le p_{(2)} \le \ldots \le p_{(m)}$
2. For each rank $k$, compute threshold: $\frac{k}{m} \times q$ (where q is the desired FDR)
3. Find the largest $k$ where $p_{(k)} \le \frac{k}{m} \times q$
4. Reject all hypotheses with rank $\le k$

In [ ]:
from statsmodels.stats.multitest import multipletests

# Apply corrections
reject_bonf, pvals_bonf, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
reject_bh, pvals_bh, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

# Compare results
methods = {
    'No correction': sig_uncorrected,
    'Bonferroni': reject_bonf,
    'Benjamini-Hochberg (FDR)': reject_bh,
}

print(f"{'Method':<28} {'Significant':>11} {'TP':>6} {'FP':>6} {'FN':>6} {'FDR':>8} {'Sensitivity':>12}")
print("-" * 85)
for name, rejected in methods.items():
    tp = (rejected & is_true_de).sum()
    fp = (rejected & ~is_true_de).sum()
    fn = (~rejected & is_true_de).sum()
    total_sig = rejected.sum()
    fdr = fp / max(total_sig, 1)
    sensitivity = tp / n_true_de
    print(f"{name:<28} {total_sig:>11} {tp:>6} {fp:>6} {fn:>6} {fdr:>8.3f} {sensitivity:>12.3f}")

In [ ]:
# Visualize the BH procedure
sorted_indices = np.argsort(p_values)
sorted_p = p_values[sorted_indices]
ranks = np.arange(1, len(sorted_p) + 1)
bh_threshold = ranks / len(sorted_p) * 0.05

fig, ax = plt.subplots(figsize=(10, 6))

# Zoom in on the first 2000 genes for visibility
n_show = 2000
ax.scatter(ranks[:n_show], sorted_p[:n_show], s=2, alpha=0.5,
           c=['red' if is_true_de[sorted_indices[i]] else 'gray' for i in range(n_show)])
ax.plot(ranks[:n_show], bh_threshold[:n_show], 'b-', linewidth=2, label='BH threshold (FDR=0.05)')
ax.set_xlabel('Rank')
ax.set_ylabel('p-value')
ax.set_title('Benjamini-Hochberg Procedure\n(red = true DE genes, gray = null genes)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Parametric vs Non-parametric Tests

### When to use which?

| Situation | Parametric test | Non-parametric alternative |
|-----------|----------------|---------------------------|
| Two independent groups, continuous data | Independent t-test | Mann-Whitney U |
| Two paired groups | Paired t-test | Wilcoxon signed-rank |
| 3+ independent groups | One-way ANOVA | Kruskal-Wallis |
| Association between categorical variables | -- | Chi-squared / Fisher's exact |

**Use parametric tests when:**
- Data is approximately normally distributed (or n > 30 by CLT)
- Variances are roughly equal between groups
- Data is continuous and measured on an interval/ratio scale

**Use non-parametric tests when:**
- Small sample sizes
- Skewed distributions, outliers
- Ordinal data or ranks
- Normality assumption clearly violated

In [ ]:
# Demonstrate: when do parametric and non-parametric tests disagree?
np.random.seed(42)

# Case 1: Normal data (both tests should agree)
normal_a = np.random.normal(10, 2, 25)
normal_b = np.random.normal(12, 2, 25)

# Case 2: Skewed data with outliers (non-parametric more robust)
skewed_a = np.random.exponential(2, 25)
skewed_b = np.random.exponential(3, 25)
# Add outliers to group a
skewed_a[0] = 50
skewed_a[1] = 45

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (a, b), title in zip(axes,
                              [(normal_a, normal_b), (skewed_a, skewed_b)],
                              ['Normal data', 'Skewed data with outliers']):
    t_stat, t_p = stats.ttest_ind(a, b)
    u_stat, u_p = stats.mannwhitneyu(a, b, alternative='two-sided')
    
    bp = ax.boxplot([a, b], labels=['Group A', 'Group B'], widths=0.5)
    ax.set_title(f"{title}\nt-test p={t_p:.4f} | Mann-Whitney p={u_p:.4f}")
    ax.set_ylabel('Value')

plt.tight_layout()
plt.show()

print("With normal data, both tests give similar p-values.")
print("With skewed/outlier data, the t-test may be misleading while Mann-Whitney is robust.")

---
## 6. Common Statistical Tests in Detail

### 6.1 Two-group comparisons

In [ ]:
# Scenario: comparing gene expression between tumor and normal samples
np.random.seed(42)

normal_expr = np.random.normal(7.5, 1.8, 40)    # normal tissue
tumor_expr = np.random.normal(9.0, 2.0, 35)     # tumor tissue

# t-test (assumes normality, roughly equal variance)
t_stat, t_p = stats.ttest_ind(tumor_expr, normal_expr)

# Welch's t-test (does not assume equal variance)
tw_stat, tw_p = stats.ttest_ind(tumor_expr, normal_expr, equal_var=False)

# Mann-Whitney U (non-parametric)
u_stat, u_p = stats.mannwhitneyu(tumor_expr, normal_expr, alternative='two-sided')

print("=== Comparing Tumor vs Normal Expression ===")
print(f"Normal: n={len(normal_expr)}, mean={normal_expr.mean():.2f}, std={normal_expr.std():.2f}")
print(f"Tumor:  n={len(tumor_expr)}, mean={tumor_expr.mean():.2f}, std={tumor_expr.std():.2f}")
print(f"\nStudent's t-test: t={t_stat:.3f}, p={t_p:.4e}")
print(f"Welch's t-test:   t={tw_stat:.3f}, p={tw_p:.4e}")
print(f"Mann-Whitney U:   U={u_stat:.0f}, p={u_p:.4e}")

### 6.2 Multi-group comparisons: ANOVA and Kruskal-Wallis

In [ ]:
# Scenario: gene expression across cancer subtypes
np.random.seed(42)

subtype_a = np.random.normal(8, 1.5, 30)    # Luminal A
subtype_b = np.random.normal(10, 2.0, 25)   # HER2+
subtype_c = np.random.normal(11, 1.8, 20)   # Triple-negative

# One-way ANOVA (parametric: assumes normality, equal variance)
f_stat, anova_p = stats.f_oneway(subtype_a, subtype_b, subtype_c)

# Kruskal-Wallis (non-parametric alternative)
h_stat, kw_p = stats.kruskal(subtype_a, subtype_b, subtype_c)

print("=== Gene Expression Across Cancer Subtypes ===")
print(f"Luminal A:       n={len(subtype_a)}, mean={subtype_a.mean():.2f}")
print(f"HER2+:           n={len(subtype_b)}, mean={subtype_b.mean():.2f}")
print(f"Triple-negative: n={len(subtype_c)}, mean={subtype_c.mean():.2f}")
print(f"\nOne-way ANOVA:   F={f_stat:.3f}, p={anova_p:.4e}")
print(f"Kruskal-Wallis:  H={h_stat:.3f}, p={kw_p:.4e}")

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([subtype_a, subtype_b, subtype_c],
                labels=['Luminal A', 'HER2+', 'Triple-neg'], widths=0.5)
ax.set_ylabel('Log2 expression')
ax.set_title(f'Expression by Cancer Subtype\nANOVA p={anova_p:.2e} | Kruskal-Wallis p={kw_p:.2e}')
plt.tight_layout()
plt.show()

### 6.3 Categorical tests: Fisher's exact and Chi-squared

These tests are used when data is categorical -- for example, testing whether a particular mutation is associated with drug response.

In [ ]:
# Scenario: is mutation X associated with drug response?
# Contingency table:
#                 Responder   Non-responder
# Mutated            35           15
# Wild-type          20           30

contingency = np.array([[35, 15],
                         [20, 30]])

# Fisher's exact test (works well for small samples)
odds_ratio, fisher_p = stats.fisher_exact(contingency)

# Chi-squared test (for larger samples)
chi2, chi2_p, dof, expected = stats.chi2_contingency(contingency)

print("=== Mutation X vs Drug Response ===")
print("Contingency table:")
table_df = pd.DataFrame(contingency,
                        index=['Mutated', 'Wild-type'],
                        columns=['Responder', 'Non-responder'])
print(table_df)
print(f"\nFisher's exact: OR={odds_ratio:.2f}, p={fisher_p:.4e}")
print(f"Chi-squared:    chi2={chi2:.3f}, dof={dof}, p={chi2_p:.4e}")
print(f"\nExpected counts (if no association):")
print(pd.DataFrame(expected.round(1),
                   index=['Mutated', 'Wild-type'],
                   columns=['Responder', 'Non-responder']))

---
## 7. Effect Sizes and Power Analysis

### 7.1 Effect size

A p-value tells you whether an effect exists, not how large it is. **Effect size** measures the magnitude of a difference.

**Cohen's d** (for two-group comparisons):

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}$$

| Cohen's d | Interpretation |
|-----------|---------------|
| 0.2 | Small |
| 0.5 | Medium |
| 0.8 | Large |

In [ ]:
def cohens_d(group1, group2):
    """Compute Cohen's d for two independent groups."""
    n1, n2 = len(group1), len(group2)
    s_pooled = np.sqrt(((n1 - 1) * group1.std(ddof=1)**2 + (n2 - 1) * group2.std(ddof=1)**2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / s_pooled

# Effect size for our tumor vs normal comparison
d = cohens_d(tumor_expr, normal_expr)
print(f"Cohen's d (Tumor vs Normal): {d:.3f}")
print(f"Interpretation: {'small' if abs(d) < 0.5 else 'medium' if abs(d) < 0.8 else 'large'} effect")

### 7.2 Power analysis

**Statistical power** is the probability of detecting a true effect. It depends on:
- **Effect size**: larger effects are easier to detect
- **Sample size**: more samples = more power
- **Significance level (alpha)**: relaxing alpha increases power but also false positives
- **Variance**: less noisy data = more power

A common target is **80% power** (beta = 0.2).

In [ ]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()

# How many samples do we need to detect various effect sizes?
effect_sizes = [0.2, 0.5, 0.8, 1.0, 1.5]
alpha = 0.05
target_power = 0.8

print(f"Sample size needed per group (power={target_power}, alpha={alpha}):")
print(f"{'Effect size (d)':<18} {'n per group':>12}")
print("-" * 32)
for d in effect_sizes:
    n = power_analysis.solve_power(effect_size=d, alpha=alpha, power=target_power)
    print(f"{d:<18.1f} {n:>12.0f}")

# Power curve
fig, ax = plt.subplots(figsize=(10, 6))
sample_sizes = np.arange(5, 200)

for d in [0.2, 0.5, 0.8, 1.0]:
    powers = [power_analysis.power(effect_size=d, nobs1=n, alpha=0.05) for n in sample_sizes]
    ax.plot(sample_sizes, powers, linewidth=2, label=f'd = {d}')

ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Power = 0.80')
ax.set_xlabel('Sample size per group')
ax.set_ylabel('Statistical power')
ax.set_title('Power Curves for Independent t-test')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## 8. Bayesian Thinking in Bioinformatics

### The concept

Frequentist statistics asks: "How likely is this data given H0?" (p-value)  
Bayesian statistics asks: "How likely is the hypothesis given the data?" (posterior)

**Bayes' theorem:**

$$P(H|D) = \frac{P(D|H) \times P(H)}{P(D)}$$

- $P(H)$ = **prior**: what we believe before seeing data
- $P(D|H)$ = **likelihood**: probability of data given hypothesis
- $P(H|D)$ = **posterior**: updated belief after seeing data

### Why it matters in bioinformatics

Consider variant calling: if a genomic position has 3 reads showing an alternative allele out of 30 total reads, is it a real variant?

- **Prior**: Most positions in the genome are NOT variants (prior probability of variant is ~0.001)
- **Likelihood**: What is the probability of 3/30 alt reads given a true variant vs. sequencing error?
- **Posterior**: Combining these gives the actual probability of a variant

In [ ]:
# Bayesian variant calling: simplified example
def bayesian_variant_call(alt_reads, total_reads, prior_variant=0.001, error_rate=0.01):
    """Compute posterior probability that a position is a true variant."""
    # Likelihood of data given true variant (heterozygous: expect ~50% alt)
    likelihood_variant = stats.binom.pmf(alt_reads, total_reads, 0.5)
    
    # Likelihood of data given no variant (sequencing error only)
    likelihood_no_variant = stats.binom.pmf(alt_reads, total_reads, error_rate)
    
    # Bayes' theorem
    prior_no_variant = 1 - prior_variant
    evidence = likelihood_variant * prior_variant + likelihood_no_variant * prior_no_variant
    posterior = (likelihood_variant * prior_variant) / evidence
    
    return posterior

# Test with different numbers of alt reads
total = 30
alt_range = range(0, 21)
posteriors = [bayesian_variant_call(a, total) for a in alt_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(alt_range, posteriors, color='steelblue', edgecolor='white')
ax.axhline(0.5, color='red', linestyle='--', label='50% threshold')
ax.set_xlabel('Number of alternative reads (out of 30)')
ax.set_ylabel('Posterior probability of true variant')
ax.set_title('Bayesian Variant Calling\n(prior P(variant) = 0.001, error rate = 1%)')
ax.legend()
plt.tight_layout()
plt.show()

print("Alt reads -> Posterior probability of variant:")
for a in [0, 1, 2, 3, 5, 10, 15]:
    print(f"  {a:2d} / {total} reads: P(variant|data) = {bayesian_variant_call(a, total):.6f}")

---
## 9. Survival Analysis Basics

**Survival analysis** studies time-to-event data -- most commonly, patient survival in clinical studies. It handles a unique challenge: **censoring**, where some subjects leave the study before the event occurs.

### Kaplan-Meier estimator

The KM estimator computes the probability of surviving past time $t$:

$$\hat{S}(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right)$$

where $d_i$ is the number of events at time $t_i$ and $n_i$ is the number at risk.

### Log-rank test

Compares survival curves between groups (e.g., mutated vs. wild-type patients).

In [ ]:
# Implement Kaplan-Meier from scratch for educational purposes
def kaplan_meier(times, events):
    """Compute Kaplan-Meier survival curve.
    
    times: array of observation times
    events: array of 1 (event occurred) or 0 (censored)
    Returns: (unique_times, survival_probs)
    """
    df = pd.DataFrame({'time': times, 'event': events}).sort_values('time')
    unique_times = sorted(df['time'].unique())
    
    n_at_risk = len(df)
    survival = 1.0
    km_times = [0]
    km_surv = [1.0]
    
    for t in unique_times:
        at_time = df[df['time'] == t]
        d_i = at_time['event'].sum()       # events at this time
        c_i = len(at_time) - d_i           # censored at this time
        
        if d_i > 0:
            survival *= (1 - d_i / n_at_risk)
        
        km_times.append(t)
        km_surv.append(survival)
        n_at_risk -= (d_i + c_i)
    
    return np.array(km_times), np.array(km_surv)

print("Kaplan-Meier estimator defined.")

In [ ]:
# Simulate survival data: two patient groups
np.random.seed(42)

# Group 1: TP53 mutant (worse prognosis)
n1 = 60
times_mut = np.random.exponential(24, n1)     # median ~24 months
censored_mut = np.random.random(n1) < 0.2     # 20% censored
events_mut = (~censored_mut).astype(int)

# Group 2: TP53 wild-type (better prognosis)
n2 = 60
times_wt = np.random.exponential(48, n2)      # median ~48 months
censored_wt = np.random.random(n2) < 0.3      # 30% censored
events_wt = (~censored_wt).astype(int)

# Cap observation at 60 months
max_time = 60
events_mut[times_mut > max_time] = 0
times_mut = np.minimum(times_mut, max_time)
events_wt[times_wt > max_time] = 0
times_wt = np.minimum(times_wt, max_time)

# Compute KM curves
km_t_mut, km_s_mut = kaplan_meier(times_mut, events_mut)
km_t_wt, km_s_wt = kaplan_meier(times_wt, events_wt)

fig, ax = plt.subplots(figsize=(10, 6))
ax.step(km_t_mut, km_s_mut, where='post', color='red', linewidth=2, label='TP53 mutant')
ax.step(km_t_wt, km_s_wt, where='post', color='blue', linewidth=2, label='TP53 wild-type')

# Mark censored events
cens_times_mut = times_mut[events_mut == 0]
cens_times_wt = times_wt[events_wt == 0]
for ct in cens_times_mut:
    idx = np.searchsorted(km_t_mut, ct, side='right') - 1
    ax.plot(ct, km_s_mut[idx], '|', color='red', markersize=8, alpha=0.5)
for ct in cens_times_wt:
    idx = np.searchsorted(km_t_wt, ct, side='right') - 1
    ax.plot(ct, km_s_wt[idx], '|', color='blue', markersize=8, alpha=0.5)

ax.set_xlabel('Time (months)')
ax.set_ylabel('Survival probability')
ax.set_title('Kaplan-Meier Survival Curves by TP53 Status')
ax.legend(loc='lower left')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Log-rank test implementation
def log_rank_test(times1, events1, times2, events2):
    """Perform log-rank test comparing two survival curves."""
    # Combine all unique event times
    all_times = np.sort(np.unique(np.concatenate([
        times1[events1 == 1], times2[events2 == 1]
    ])))
    
    observed_1 = 0  # total observed events in group 1
    expected_1 = 0  # total expected events in group 1
    var_sum = 0
    
    for t in all_times:
        # At risk in each group
        n1 = np.sum(times1 >= t)
        n2 = np.sum(times2 >= t)
        n = n1 + n2
        
        # Events in each group
        d1 = np.sum((times1 == t) & (events1 == 1))
        d2 = np.sum((times2 == t) & (events2 == 1))
        d = d1 + d2
        
        if n > 1:
            expected_1 += d * n1 / n
            observed_1 += d1
            var_sum += d * n1 * n2 * (n - d) / (n * n * (n - 1))
    
    # Chi-squared statistic with 1 df
    chi2 = (observed_1 - expected_1) ** 2 / var_sum if var_sum > 0 else 0
    p_value = 1 - stats.chi2.cdf(chi2, df=1)
    
    return chi2, p_value

chi2, p_val = log_rank_test(times_mut, events_mut, times_wt, events_wt)
print(f"Log-rank test: chi2 = {chi2:.3f}, p = {p_val:.4e}")
print(f"The survival difference is {'statistically significant' if p_val < 0.05 else 'not significant'}.")

---
## 10. Practical: Complete Statistical Analysis Workflow

Let us apply everything to a realistic scenario: analyzing a gene expression dataset across treatment conditions.

In [ ]:
# Simulated RNA-seq-like dataset: 1000 genes, 3 conditions, 10 replicates each
np.random.seed(42)

n_genes = 1000
n_per_group = 10
conditions = ['Control', 'Drug_A', 'Drug_B']

# Most genes are not differentially expressed
n_de_a = 50    # DE in Drug A vs Control
n_de_b = 80    # DE in Drug B vs Control
n_de_both = 20 # DE in both

gene_names = [f'Gene_{i:04d}' for i in range(n_genes)]

# Generate baseline expression
baseline = np.random.normal(8, 3, n_genes)
baseline = np.clip(baseline, 1, 15)  # realistic range

# Generate expression for each condition
data = {}
for cond in conditions:
    for rep in range(n_per_group):
        col_name = f"{cond}_rep{rep+1}"
        expr = baseline + np.random.normal(0, 0.5, n_genes)  # biological noise
        data[col_name] = expr

expr_df = pd.DataFrame(data, index=gene_names)

# Add differential expression effects
de_a_idx = list(range(n_de_a))
de_b_idx = list(range(n_de_a - n_de_both, n_de_a - n_de_both + n_de_b))

for idx in de_a_idx:
    effect = np.random.choice([-1, 1]) * np.random.uniform(1, 3)
    for rep in range(n_per_group):
        expr_df.iloc[idx, n_per_group + rep] += effect  # Drug A columns

for idx in de_b_idx:
    effect = np.random.choice([-1, 1]) * np.random.uniform(1, 3)
    for rep in range(n_per_group):
        expr_df.iloc[idx, 2 * n_per_group + rep] += effect  # Drug B columns

print(f"Expression matrix: {expr_df.shape[0]} genes x {expr_df.shape[1]} samples")
print(f"True DE genes in Drug A: {n_de_a}")
print(f"True DE genes in Drug B: {n_de_b}")
print(f"True DE in both: {n_de_both}")
expr_df.head()

In [ ]:
# Differential expression analysis: Drug A vs Control
control_cols = [c for c in expr_df.columns if c.startswith('Control')]
drug_a_cols = [c for c in expr_df.columns if c.startswith('Drug_A')]
drug_b_cols = [c for c in expr_df.columns if c.startswith('Drug_B')]

results = []

for gene in expr_df.index:
    ctrl = expr_df.loc[gene, control_cols].values
    da = expr_df.loc[gene, drug_a_cols].values
    db = expr_df.loc[gene, drug_b_cols].values
    
    # Drug A vs Control
    t_a, p_a = stats.ttest_ind(da, ctrl)
    fc_a = da.mean() - ctrl.mean()  # log2 fold change
    
    # Drug B vs Control
    t_b, p_b = stats.ttest_ind(db, ctrl)
    fc_b = db.mean() - ctrl.mean()
    
    results.append({
        'gene': gene,
        'logFC_A': fc_a, 'pvalue_A': p_a,
        'logFC_B': fc_b, 'pvalue_B': p_b,
    })

results_df = pd.DataFrame(results).set_index('gene')

# Apply FDR correction
_, results_df['padj_A'], _, _ = multipletests(results_df['pvalue_A'], method='fdr_bh')
_, results_df['padj_B'], _, _ = multipletests(results_df['pvalue_B'], method='fdr_bh')

print(f"Significant genes (FDR < 0.05):")
print(f"  Drug A vs Control: {(results_df['padj_A'] < 0.05).sum()}")
print(f"  Drug B vs Control: {(results_df['padj_B'] < 0.05).sum()}")
print(f"  Both: {((results_df['padj_A'] < 0.05) & (results_df['padj_B'] < 0.05)).sum()}")
results_df.head(10)

In [ ]:
# Volcano plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, fc_col, padj_col, title in zip(
    axes,
    ['logFC_A', 'logFC_B'],
    ['padj_A', 'padj_B'],
    ['Drug A vs Control', 'Drug B vs Control']
):
    log_p = -np.log10(results_df[padj_col].clip(lower=1e-50))
    sig = results_df[padj_col] < 0.05
    
    ax.scatter(results_df.loc[~sig, fc_col], log_p[~sig],
              s=5, alpha=0.5, color='gray', label='Not significant')
    ax.scatter(results_df.loc[sig, fc_col], log_p[sig],
              s=15, alpha=0.7, color='red', label='FDR < 0.05')
    ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5)
    ax.axvline(-1, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(1, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Log2 fold change')
    ax.set_ylabel('-Log10(adjusted p-value)')
    ax.set_title(title)
    ax.legend(markerscale=3)

plt.suptitle('Volcano Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Summary

| Topic | Key takeaway |
|-------|-------------|
| **Distributions** | Choose tests that match your data's distribution |
| **Multiple testing** | Always correct for multiple comparisons in genomics (BH/FDR preferred) |
| **Parametric vs non-parametric** | Use non-parametric when assumptions are violated |
| **Effect sizes** | Report alongside p-values; a significant p-value can have tiny effect |
| **Power** | Plan sample sizes before experiments; underpowered studies waste resources |
| **Bayesian** | Incorporates prior knowledge; natural for variant calling |
| **Survival analysis** | Handles censored data; essential for clinical bioinformatics |

---
## Exercises

### Exercise 1: Multiple Testing Simulation

Simulate testing 10,000 genes where:
- 200 genes are truly differentially expressed (effect size d=1.0)
- 9,800 genes have no effect (null)
- Each gene has 15 samples per group

For each gene, generate data from normal distributions and run a t-test. Then:
1. Apply Bonferroni and BH corrections
2. Count TP, FP, FN for each method
3. Compute the achieved FDR and sensitivity
4. Create a comparison table

In [ ]:
# Your solution here


### Exercise 2: Choose the Right Test

For each scenario below, state which statistical test is most appropriate and why. Then run the test using the provided data.

**Scenario A:** You measured protein levels (continuous, normally distributed) in matched tumor/normal pairs from 20 patients. Test whether tumor has higher levels.

**Scenario B:** You have mutation counts (integers, overdispersed) in three genomic regions. Each region has 50 genes. Test whether mutation rates differ between regions.

**Scenario C:** In a study of 200 patients, you observe this breakdown:
- BRCA1 mutant + early onset: 45
- BRCA1 mutant + late onset: 15  
- BRCA1 wild-type + early onset: 60
- BRCA1 wild-type + late onset: 80

Test whether BRCA1 mutation is associated with early onset.

In [ ]:
# Scenario A: paired data
np.random.seed(42)
normal_levels = np.random.normal(5, 1.5, 20)
tumor_levels = normal_levels + np.random.normal(0.8, 0.5, 20)  # paired increase

# Your test here


In [ ]:
# Scenario B: count data, 3 groups
np.random.seed(42)
region_1 = np.random.negative_binomial(3, 0.3, 50)
region_2 = np.random.negative_binomial(5, 0.3, 50)
region_3 = np.random.negative_binomial(3, 0.3, 50)

# Your test here


In [ ]:
# Scenario C: contingency table
table = np.array([[45, 15], [60, 80]])

# Your test here


### Exercise 3: Power Analysis for Experimental Design

You are planning an RNA-seq experiment to detect differentially expressed genes between a knockout (KO) and wild-type (WT) mouse. Based on pilot data, you expect:
- Standard deviation of log2 expression: ~1.5
- Minimum biologically meaningful fold change: 1.5x (log2 FC = 0.585)
- This gives Cohen's d = 0.585 / 1.5 = 0.39

Tasks:
1. How many mice per group do you need for 80% power at alpha=0.05?
2. Plot the power curve for sample sizes from 5 to 100
3. How does the required sample size change if you relax your FC threshold to 2x (log2 FC = 1.0)?
4. If you can only afford 10 mice per group, what is the minimum detectable effect size at 80% power?

In [ ]:
# Your solution here


### Exercise 4: Bayesian Analysis of Mutation Burden

A cancer gene panel tests 50 genes. You observe that a patient has non-synonymous mutations in 8 of 50 genes.

Your prior beliefs:
- Background mutation rate per gene: 3% (i.e., probability of mutation in any given gene)
- "High mutation burden" phenotype has a rate of 25% per gene
- Prior probability that this patient has high mutation burden: 10%

Tasks:
1. Compute the likelihood of observing 8/50 mutations under each hypothesis
2. Compute the posterior probability of high mutation burden
3. How does the posterior change as you observe 1, 2, ..., 20 mutations?
4. Plot the prior-to-posterior update

In [ ]:
# Your solution here


### Exercise 5: Survival Analysis with Gene Expression

Simulate a clinical dataset with 120 patients:
- Generate a continuous gene expression value for each patient from `np.random.normal(10, 3, 120)`
- Split patients into "high" and "low" expression groups using the median
- For "high" group: survival times from `np.random.exponential(36, n_high)` (worse prognosis)
- For "low" group: survival times from `np.random.exponential(60, n_low)` (better prognosis)
- Censor 25% of patients randomly, cap all times at 72 months

Tasks:
1. Plot KM curves for both groups
2. Perform the log-rank test
3. Report the median survival time for each group
4. What happens if you split at the 25th vs 75th percentile instead of the median?

In [ ]:
# Your solution here


### Exercise 6: Complete DE Analysis Pipeline

Using the `expr_df` dataset from Section 10:

1. Run a one-way ANOVA for each gene across all 3 conditions (Control, Drug A, Drug B)
2. Apply BH correction to the ANOVA p-values
3. For significant genes (FDR < 0.05), run pairwise t-tests (Control vs Drug A, Control vs Drug B, Drug A vs Drug B) and apply BH correction to each set
4. Create a Venn-diagram-like summary: how many genes are affected by Drug A only, Drug B only, or both?
5. For the top 10 most significant genes (by ANOVA p-value), create a heatmap showing expression across all 30 samples

In [ ]:
# Your solution here


---[< Previous: Promoter and Regulatory Sequence Analysis](../05_Promoter_and_Regulatory_Analysis/02_regulatory_analysis.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Machine Learning for Biology >](../07_Machine_Learning_for_Biology/01_machine_learning_for_biology.ipynb)---